# Possession Lens

Frame-level football possession with a recurrent neural network.

**Authors:** Hrishikesh Karmalkar, Basel, Omar, Michail and Paul.

Run cells from top to bottom. Downloads, training and match inference are off by default; enable each deliberately below. All training/tuning uses external data. Never compare competition predictions against its video or ground truth. Individual track slots are not verified players. Clear notebook outputs before publishing.

In [ ]:
from pathlib import Path
import sys, json, subprocess
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "controlswitch").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the possession-lens project"
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
print("Project:", ROOT.name)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)

## 1. Configuration

Set your data paths. The bundled checkpoint can run inference without training again. The initial local external match remains available as a parsed cache; a fresh checkout must download external data separately.

In [ ]:
DATA_SOURCE = ROOT / "data/raw/MORE THAN A HACK"
DATA_ROOT = ROOT / "data/raw/provided"
EXTERNAL_ROOT = ROOT / "work/external/raw"
EXTERNAL_CACHE = ROOT / "work/external/cache"
MODEL = ROOT / "models/rnn-symmetric-001.npz"
RUN_DOWNLOADS = False
RUN_TRAINING = False
RUN_INFERENCE = False
ALLOW_SINGLE_MATCH_PILOT = False

def run_cli(*args):
    subprocess.run([sys.executable, "-m", "possession_lens", *map(str,args)], cwd=ROOT, check=True)

## 2. Organise licensed files locally

This creates file links in the ignored data folder. Original files stay untouched. Nothing is uploaded.

In [ ]:
if DATA_SOURCE.is_dir():
    from controlswitch.data_setup import organise
    layout = organise(DATA_SOURCE, DATA_ROOT)
    print("Local linked assets:", len(layout["assets"]))
else:
    print("Set DATA_SOURCE to your authorised organiser downloads before running inference.")

## 3. Download external data

IDSSE/Sportec data is CC BY 4.0: Bassek et al. (2025), doi:10.1038/s41597-025-04505-y. The required three-match files total about 1.1 GB. Downloads may be unavailable in restricted environments. No training can proceed on the provided competition data as a substitute.

In [ ]:
if RUN_DOWNLOADS:
    run_cli("download-external", "--out", EXTERNAL_ROOT)
else:
    print("External downloads disabled. Set RUN_DOWNLOADS=True when ready.")

## 4. Train the RNN on external matches only

Default: separate training and validation matches. An explicit single-match pilot is weaker evidence. The previously viewed test half must not be reused as a fresh test. Training uses a 24-unit tanh RNN with truncated backpropagation through time.

In [ ]:
if RUN_TRAINING:
    from datetime import datetime
    train_out = ROOT / "work/rnn" / ("notebook-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    args = ["rnn-train", "--external-root", EXTERNAL_ROOT, "--out", train_out, "--epochs", "5"]
    if ALLOW_SINGLE_MATCH_PILOT:
        args.append("--single-match-pilot")
    run_cli(*args)
    MODEL = train_out / "rnn-model.npz"
else:
    print("Training disabled; selected checkpoint:", MODEL.name)

## 5. Inspect external evidence

The latest symmetric model has one independent external test (J03WN1). Historical reports remain available. J03WMX and J03WN1 are now viewed tests; do not use either for further tuning.

In [ ]:
report_path = ROOT / "models/rnn-symmetric-001-report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    rows = []
    for name, metrics in report["fresh_match"]["test_results"].items():
        rows.append({"model": name, "clean_spell_f1": metrics["clean"]["spell_f1_mean"], "masked_spell_f1": metrics["masked"]["spell_f1_mean"], "clean_predicted_spells": metrics["clean"]["n_pred_spells"]})
    display(pd.DataFrame(rows))
    print("Fresh external match:", report["fresh_match"]["protocol"]["match"]["title"])
    print("No competition accuracy measured.")


## 6. Frozen inference on the provided match

This only applies weights trained externally. No fitting, calibration or output-versus-video check is performed. Output folders must be new and empty.

In [ ]:
if RUN_INFERENCE:
    from datetime import datetime
    prediction_out = ROOT / "work/rnn" / ("inference-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    run_cli("rnn-predict", "--data-root", DATA_ROOT, "--model", MODEL, "--out", prediction_out)
    stats = json.loads((prediction_out / "team-statistics.json").read_text())
    display(pd.DataFrame([{k: stats[k] for k in ["home_possession_percent", "away_possession_percent", "live_seconds", "dead_seconds", "unknown_seconds"]}]))
else:
    print("Inference disabled. Set RUN_INFERENCE=True to create a new local prediction run.")

## 7. Tests and next benchmark

Team percentage denominators exclude dead/unknown frames. Tracklet proxies must retain unassigned time. Run the tests before sharing changes. Additional-match model comparisons require fresh external files and a frozen test protocol.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=ROOT, check=True)

## GitHub sharing

Share source, this cleared notebook, pinned dependencies, authors, model card, checkpoints and external aggregate reports. Do not share `data/`, `work/`, private local manifests, videos or competition prediction timelines. The judge website demo uses synthetic data.

## Reduced-switching RNN
See `RNN_IMPROVEMENT.md` for measured validation results and limits. The improved checkpoint is `models/rnn-decoded-001.npz`. Its decoder uses future evidence for offline analysis. Three-match evaluation requires complete J03WPY and J03WMX XML files in addition to J03WOH training data. Keep the switches below disabled until those files are available; score the fresh test once after model selection is final.

In [ ]:
import json
switching_report = json.loads((ROOT / "models/rnn-decoded-001-report.json").read_text())
print("Selected RNN:", switching_report["selected_rnn"])
print("Selected baseline:", switching_report["selected_baseline"])
print("Fresh-test results:", switching_report["test_results"])
RUN_THREE_MATCH_BENCHMARK = False
EVALUATE_FRESH_TEST = False
if RUN_THREE_MATCH_BENCHMARK:
    from controlswitch.rnn_benchmark import run
    benchmark_report = run(
        [ROOT / "work/external/raw", ROOT / "work/external/cache"],
        ROOT / "work/rnn-three-match-001", epochs=5,
        evaluate_test=EVALUATE_FRESH_TEST)


## Current model and development status
The default checkpoint is `models/rnn-symmetric-001.npz`. It uses unchanged recurrent weights with exact team-permutation averaging. The fourth-match result and failed development trials are in `RNN_SYMMETRY.md` and `models/rnn-symmetric-001-report.json`. The candidate was not re-tested on J03WMX. Further changes require a different independent match; reproducing an existing test is not new evidence.